# Live File-Duplicate Detection with Spark

This Structured Streaming example watches a folder for new files, computes each file's SHA-256 hash, and prints hashes found in more than one file. Files with identical contents have the same hash. On the first run, the code creates three sample text files, two of which are duplicates.

The query uses a checkpoint to resume file progress and aggregation state. The console sink is for demonstration; it prints the current duplicate groups. Run this notebook with PySpark and Java configured.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, collect_list, count, sha2

base_dir = Path.cwd() / "duplicate_stream_demo"
input_dir = base_dir / "input"
checkpoint_dir = base_dir / "checkpoint"
input_dir.mkdir(parents=True, exist_ok=True)

# Seed the input directory with duplicate and non-duplicate sample files once.
if not any(input_dir.iterdir()):
    (input_dir / "report-a.txt").write_text("same file contents\n", encoding="utf-8")
    (input_dir / "report-copy.txt").write_text("same file contents\n", encoding="utf-8")
    (input_dir / "notes.txt").write_text("different contents\n", encoding="utf-8")

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("LiveDuplicateFileDetection")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

streaming_files = (
    spark.readStream
    .format("binaryFile")
    .option("maxFilesPerTrigger", 1)
    .load(input_dir.as_uri())
    .select(
        col("path"),
        sha2(col("content"), 256).alias("sha256")
    )
)

duplicate_groups = (
    streaming_files
    .groupBy("sha256")
    .agg(
        count("*").alias("copies"),
        collect_list("path").alias("file_paths")
    )
    .where(col("copies") > 1)
)

query = (
    duplicate_groups.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", checkpoint_dir.as_uri())
    .start()
)

print("Watching for files in:", input_dir)
print("Duplicate results appear in the streaming console output.")

To test live detection, copy another file into `duplicate_stream_demo/input` while the query is running. A file with the same contents as an existing file will have the same SHA-256 hash and appear in a duplicate group. Check `query.status` to see whether the query is active; run `query.stop()` to stop it cleanly. Keep the checkpoint folder when restarting the same query.